In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
stations = gpd.read_file("../data/raw/miteco_stations/Estaciones_Evaluacion_2024.shp")

In [3]:
stations.shape

(4641, 14)

In [4]:
stations.columns.tolist()

['ANNO',
 'COD_NACION',
 'COD_ESTACI',
 'LATITUD_G',
 'LONGITUD_G',
 'CONTAMINAN',
 'NOMBRE',
 'VALOR_LEGI',
 'ESTADISTIC',
 'VALOR',
 'SITUACION',
 'LEYENDA',
 'COMENTARIO',
 'geometry']

In [5]:
stations.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [6]:
stations.head()

,ANNO,COD_NACION,COD_ESTACI,LATITUD_G,LONGITUD_G,CONTAMINAN,NOMBRE,VALOR_LEGI,ESTADISTIC,VALOR,SITUACION,LEYENDA,COMENTARIO,geometry
0,2024.0,1022001.0,ES1672A,42.518333,-2.619444,O3,EL CIEGO,VOS,media_3años,3.0,3.0,<VO,años que partipan en la media (2022;2023;2024)...,POINT (-2.61944 42.51833)
1,2024.0,1022001.0,ES1672A,42.518333,-2.619444,PM10,EL CIEGO,VLD,percentil_90_4,15.0,1.0,<=UEI,"0 dias>50 y 98,09 % datos",POINT (-2.61944 42.51833)
2,2024.0,1022001.0,ES1672A,42.518333,-2.619444,NO2,EL CIEGO,VLH,percentil_99_8,19.0,1.0,<=UEI,"0 horas>200 y 99,21 % datos",POINT (-2.61944 42.51833)
3,2024.0,1022001.0,ES1672A,42.518333,-2.619444,PM10,EL CIEGO,VLA,media,9.0,1.0,<=UEI,"98,09 % datos",POINT (-2.61944 42.51833)
4,2024.0,1022001.0,ES1672A,42.518333,-2.619444,NO2,EL CIEGO,VLA,media,4.5,1.0,<=UEI,"99,21 % datos",POINT (-2.61944 42.51833)


In [7]:
stations["station_code"] = stations["COD_NACION"].astype(int).astype(str).str.zfill(8)

In [8]:
stations[["COD_NACION", "station_code"]].head()

,COD_NACION,station_code
0,1022001.0,01022001
1,1022001.0,01022001
2,1022001.0,01022001
3,1022001.0,01022001
4,1022001.0,01022001


In [12]:
is_canarias = stations["station_code"].str.startswith(("35", "38"))
canarias = stations[is_canarias]
canarias_stations = canarias.drop_duplicates("station_code")
len(canarias_stations)

51

In [13]:
canarias_stations[["station_code", "COD_ESTACI", "NOMBRE", "LATITUD_G", "LONGITUD_G"]]

,station_code,COD_ESTACI,NOMBRE,LATITUD_G,LONGITUD_G
3095,35002001,ES1745A,AGÜIMES,27.897639,-15.445028
3103,35004001,ES1863A,ARRECIFE,28.976990,-13.547584
3112,35004002,ES1955A,CIUDAD DEPORTIVA-ARRECIFE,28.967870,-13.555856
3120,35006001,ES1952A,POLIDEPORTIVO AFONSO-ARUCAS,28.111266,-15.520924
3128,35016010,ES0873A,JINÁMAR III FASE,28.035120,-15.416310
3142,35016011,ES1451A,NÉSTOR ÁLAMO,28.031560,-15.409140
3150,35016012,ES1573A,MERCADO CENTRAL,28.133732,-15.432823
3159,35016014,ES2097A,SAN NICOLÁS,28.103300,-15.422800
3168,35017001,ES1865A,CENTRO DE ARTE JUAN ISMAEL,28.500255,-13.852395
3177,35017002,ES1866A,PARQUE DE LA PIEDRA,28.502762,-13.853699


In [14]:
from pathlib import Path

files = Path("../data/raw/eea_pm10").glob("*.parquet")    # los 121 archivos
pm10_codes = {f.name.split("_")[1] for f in files}        # el código de cada uno, sin repetir
len(pm10_codes)                                           # tendría que dar 56

56

In [15]:
missing = pm10_codes - set(canarias_stations["station_code"])
print(len(missing))
sorted(missing)

7


['35002002',
 '35002003',
 '35016013',
 '35019002',
 '35022002',
 '38006002',
 '38024001']

In [16]:
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
columns = ["station_code", "COD_ESTACI", "NOMBRE", "LATITUD_G", "LONGITUD_G"]
station_coordinates = canarias_stations[columns]
station_coordinates.to_csv(out_dir / "canarias_station_coordinates.csv", index=False)

In [17]:
pd.read_csv(out_dir / "canarias_station_coordinates.csv").head()

,station_code,COD_ESTACI,NOMBRE,LATITUD_G,LONGITUD_G
0,35002001,ES1745A,AGÜIMES,27.897639,-15.445028
1,35004001,ES1863A,ARRECIFE,28.976990,-13.547584
2,35004002,ES1955A,CIUDAD DEPORTIVA-ARRECIFE,28.967870,-13.555856
3,35006001,ES1952A,POLIDEPORTIVO AFONSO-ARUCAS,28.111266,-15.520924
4,35016010,ES0873A,JINÁMAR III FASE,28.035120,-15.416310
